In [ ]:
# Pipeline Orchestration

Trigger, monitor, and query pipelines via the platform API and Spark Connect.

++
||
++
++



In [ ]:
import time, textwrap, requests
from datetime import datetime
from pyspark.sql import SparkSession

# ── Config ────────────────────────────────────────────────────────────────────
API_BASE        = "http://localhost:8000/api/v1/etl"
SPARK_REMOTE    = "sc://localhost:15002"
BUSINESS_DB     = "data_20260416"
POLL_INTERVAL_S = 3
POLL_TIMEOUT_S  = 300

# ── Spark session ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
spark.sql(f"USE {BUSINESS_DB}")
print(f"Spark:  {spark.version}  →  db={BUSINESS_DB}")


# ── API helpers ───────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def _get(path, **params):
    r = session.get(f"{API_BASE}{path}", params=params or None)
    r.raise_for_status()
    return r.json()


def _post(path, body=None):
    r = session.post(f"{API_BASE}{path}", json=body)
    r.raise_for_status()
    return r.json()


def _delete(path):
    r = session.delete(f"{API_BASE}{path}")
    r.raise_for_status()


# ── Pipeline helpers ──────────────────────────────────────────────────────────
def list_pipelines(status=None):
    """Return pipelines as a list of dicts; optionally filter by status."""
    return _get("/pipelines", **({"status": status} if status else {}))


def trigger(pipeline_id: int, business_date: str | None = None) -> dict:
    """Trigger a pipeline run and return the run summary."""
    body = {"business_date": business_date} if business_date else {}
    run = _post(f"/pipelines/{pipeline_id}/run", body)
    print(f"[{datetime.now():%H:%M:%S}]  triggered  pipeline={pipeline_id}  run_id={run['id']}")
    return run


def get_run(run_id: int) -> dict:
    return _get(f"/runs/{run_id}")


def cancel_run(run_id: int) -> dict:
    result = _post(f"/runs/{run_id}/cancel")
    print(f"[{datetime.now():%H:%M:%S}]  cancelled  run_id={run_id}")
    return result


def wait_for_run(run_id: int, timeout_s: int = POLL_TIMEOUT_S, interval_s: int = POLL_INTERVAL_S) -> dict:
    """
    Poll until the run reaches a terminal state (completed / failed / cancelled).
    Returns the final RunDetail dict.  Raises RuntimeError on timeout.
    """
    terminal = {"completed", "failed", "cancelled"}
    deadline = time.monotonic() + timeout_s
    last_status = None

    while time.monotonic() < deadline:
        run = get_run(run_id)
        status = run["status"]

        if status != last_status:
            step_line = ""
            if run.get("steps"):
                step_line = "  steps: " + " → ".join(
                    f"{s['step_type']}={s['status']}" for s in run["steps"]
                )
            print(f"[{datetime.now():%H:%M:%S}]  run={run_id}  status={status}{step_line}")
            last_status = status

        if status in terminal:
            return run

        time.sleep(interval_s)

    raise RuntimeError(f"run {run_id} did not finish within {timeout_s}s")


def run_summary(run: dict):
    """Print a concise summary of a completed run."""
    steps = run.get("steps", [])
    w = 60
    print("─" * w)
    print(f"  Run {run['id']}   status={run['status'].upper()}")
    print(f"  duration   {run.get('duration_seconds', '—')} s")
    print(f"  extracted  {run.get('records_extracted', 0):,}")
    print(f"  loaded     {run.get('records_loaded', 0):,}")
    if steps:
        print("  ── steps ──")
        for s in steps:
            dur = f"{s['duration_seconds']:.1f}s" if s.get("duration_seconds") else "—"
            print(f"    {s['step_type']:<12}  {s['status']:<12}  {dur:>7}"
                  f"  in={s['records_in']:,}  out={s['records_out']:,}")
    if run.get("error_message"):
        print(f"  error: {run['error_message']}")
    print("─" * w)


print("Helpers ready.")

+-------------+---------+-----------+
|namespace    |tableName|isTemporary|
+-------------+---------+-----------+
|data_20260416|test_dw  |false      |
+-------------+---------+-----------+



In [ ]:
## 1 — Inspect pipelines

+----------+------+--------+--------+-------------+--------+--------+-------+---+----------+-----------+--------+--------------+
|DATE      |APP_ID|APP_NAME|TRADE_ID|TRADE_VERSION|CUSTOMER|PRODUCT |PRICE  |QTY|TRADE_DATE|SETTLE_DATE|BOOK    |application_id|
+----------+------+--------+--------+-------------+--------+--------+-------+---+----------+-----------+--------+--------------+
|2026-04-01|1     |Eta     |1       |VAL_4888     |VAL_5074|VAL_1969|8735.5 |656|2026-02-18|2026-07-24 |VAL_7674|2             |
|2026-03-18|2     |Delta   |2       |VAL_7104     |VAL_2862|VAL_8567|2305.98|80 |2026-12-08|2026-11-28 |VAL_1748|2             |
|2026-06-19|3     |Theta   |3       |VAL_6567     |VAL_1951|VAL_3665|2169.5 |126|2026-07-16|2026-11-01 |VAL_3746|2             |
|2026-09-12|4     |Delta   |4       |VAL_3028     |VAL_8413|VAL_1654|7679.9 |159|2026-02-28|2026-01-01 |VAL_5811|2             |
|2026-03-17|5     |Gamma   |5       |VAL_8617     |VAL_3988|VAL_5711|1068.51|72 |2026-02-13|2026-

In [ ]:
pipelines = list_pipelines()
for p in pipelines:
    print(f"  id={p['id']:<4}  {p['name']:<40}  status={p['status']}")

## 2 — Trigger a pipeline and wait

In [ ]:
# ── Configure ─────────────────────────────────────────────────────────────────
PIPELINE_ID    = pipelines[0]["id"]   # change to the target pipeline id
BUSINESS_DATE  = "2026-04-16"         # or None for the platform default

# ── Run ───────────────────────────────────────────────────────────────────────
run = trigger(PIPELINE_ID, BUSINESS_DATE)
completed_run = wait_for_run(run["id"])
run_summary(completed_run)

## 3 — Query results via Spark

In [ ]:
TARGET_TABLE = "test_dw"   # adjust as needed

if completed_run["status"] != "completed":
    raise RuntimeError(f"Run did not complete — status={completed_run['status']}")

df = spark.sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 1000")

print(f"Schema: {TARGET_TABLE}")
df.printSchema()

row_count = df.count()
print(f"Rows fetched: {row_count:,}")

## 4 — Data quality checks

In [ ]:
from pyspark.sql import functions as F

# Null counts per column
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

issues = {col: n for col, n in null_counts.items() if n > 0}

if issues:
    print(f"WARNING — {len(issues)} column(s) have nulls:")
    for col, n in issues.items():
        print(f"  {col}: {n:,} nulls")
else:
    print("Quality check passed — no nulls found.")

# Row count vs records_loaded reconciliation
loaded = completed_run.get("records_loaded", 0)
print(f"\nReconciliation: API reported {loaded:,} loaded,  Spark count={row_count:,}")

## 5 — Ad-hoc control

In [ ]:
# List active (currently running) runs
active_run_ids = _get("/active")
print("Active runs:", active_run_ids or "none")

# Cancel a specific run if needed
# cancel_run(run_id=42)

# Pull recent runs for a pipeline
# recent = _get(f"/pipelines/{PIPELINE_ID}/runs", limit=5)
# for r in recent:
#     print(f"  run={r['id']}  status={r['status']}  started={r.get('started_at')}")